#### Short Term Memory

##### Imports

In [3]:
import os
from dotenv import load_dotenv
load_dotenv()
from langchain.agents import create_agent, AgentState
from langgraph.checkpoint.memory import InMemorySaver
from langchain_community.utilities import SQLDatabase

##### Initializing Database

In [2]:
DB_URL = os.getenv("DB_URL")
db = SQLDatabase.from_uri(DB_URL)

##### State Schema

In [4]:
class MyState(AgentState):
    user_name: str
    theme: str
    

##### Creating Agent

In [5]:
agent = create_agent(
    model="ollama:qwen2.5:7b",
    state_schema=MyState,
    checkpointer=InMemorySaver()
)

##### CheckPointer (Short Term Memory)

In [19]:
config = {"configurable": {"thread_id": "chat-session-1"}}

##### Invoking Agent

In [ ]:
res1 = agent.invoke(
    {
        'messages': [{"role": "user", "content": "Hi Who is Narendra Modi ?"}],
        'user_name': 'peti lata',
        'theme': 'light'
    },
    config=config,
)
print(f"{res1["messages"][-1].content}\n")

res2 = agent.invoke(
    {
        'messages': [{"role": "user", "content": "just now what i asked ?"}],
        'user_name': 'peti lata',
        'theme': 'light'
    },
    config=config,
)
print(f"{res2["messages"][-1].content}")

#### Using Postgres as checkpointer

In [ ]:
import psycopg
import psycopg_binary
from langgraph.checkpoint.postgres import PostgresSaver

with PostgresSaver.from_conn_string(DB_URL) as checkpointer:
    checkpointer.setup()
    agent = create_agent(
        model="ollama:qwen2.5:7b",
        checkpointer=checkpointer
    )
    config = {"configurable": {"thread_id": "db-session-01"}}
    fResp = agent.invoke(
        {"messages": [{"role": "user", "content": "Hi Can you show mw a road map for applied ai engineer and it's salary in india"}]},
        config=config
    )
    print(f"{fResp["messages"][-1].content}\n")
    print("+++++++++++Second Response+++++++++++++")
    
    sResp = agent.invoke(
            {"messages": [{"role": "user", "content": "What about experience developer salary in that ?"}]},
            config=config
        )
    
    print(f"{sResp["messages"][-1].content}\n")